# Prepare the ADNI baseline MCI cohort

Run cells from top to bottom. This runs the local preparation workflow, not a model. Original CSVs remain unchanged. The prediction horizon is provisionally **36 months**. The selected matching policy is **90 days before baseline, with results available by baseline**. Configuration is in `config/workflow.json`; rules and limitations are in `docs/decisions.md`.


In [ ]:
from pathlib import Path
import csv, json, sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/scripts/prepare_adni.py").is_file()), None)
assert root is not None, "Open this notebook within the final-upload project folder"
sys.path.insert(0, str(root / "src/scripts"))
from prepare_adni import main
print(json.dumps(json.loads((root / "config/workflow.json").read_text()), indent=2))


## Run preparation

This copies the raw inputs if necessary and regenerates the processed data and reports. It performs the requested participant-uniqueness, label, join-cardinality and temporal checks. Unknown outcomes stay missing.


In [ ]:
summary = main()


## Inspect file inventory and cohort flow

All displayed counts are read from this run. Identifiers are not displayed here.


In [ ]:
def show_csv(relative, columns=None, limit=40):
    with (root / relative).open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        print("No rows:", relative)
        return
    columns = columns or list(rows[0])
    print(" | ".join(columns))
    for row in rows[:limit]:
        print(" | ".join(str(row[c]) for c in columns))
    if len(rows) > limit:
        print(f"... {len(rows)-limit} further rows in {relative}")

show_csv("reports/inventory.csv", ["table", "rows", "columns", "unique_RID", "unique_PTID"])
print("\nCohort flow")
show_csv("reports/cohort_flow.csv")


## Missingness and modality coverage

“Any” means at least one selected feature per modality; “complete” requires every selected feature. Eligibility also requires a known outcome. Very small MRI/CSF groups do not justify a multimodal modeling claim.


In [ ]:
show_csv("reports/modality_coverage.csv")
print("\nPredictor missingness")
show_csv("reports/predictor_missingness.csv")


## Follow-up, phase and timing sensitivity

The timing report compares 0/30/60/90/180-day lookbacks and shows a retrospective acquisition-only alternative. These alternatives do not enter the master. MRI still requires finalized passing QC and CSF still requires verified assay methods.


In [ ]:
show_csv("reports/cohort_by_phase.csv")
show_csv("reports/followup_distribution.csv")
show_csv("reports/timing_sensitivity.csv", limit=70)


## Requested validation checks

The focused synthetic boundary cases below are software validation only. They are never added to participant datasets.


In [ ]:
show_csv("reports/validation_checks.csv")
from validate_rules import main as validate_rules
validate_rules()


## Before modeling

Confirm the 36-month horizon and the documented-event/reversion rules with the supervisor. Obtain the consolidated Elecsys methods for batches 9, 11 and 13 and a Key MRI export dictionary. Review the strict date-availability limitation: processing dates do not prove clinical release dates. Read `reports/data_quality_report.md` and `docs/decisions.md`. Keep unknown labels out of supervised fitting; keep future outcomes/provenance out of X. Use only `predictor_schema.json` allowlisted features; fit later preprocessing inside participant-separated training folds.
